# 黑马头条项目｜第09天：用户行为实战

> 学习目标：记录关注、点赞、阅读、不喜欢、收藏行为，在文章详情页回显当前用户的互动状态；理解 Redis 行为存储、身份关联、操作幂等以及 Jackson 对 Long 类型 ID 的序列化处理。

本篇依据百度网盘 `day09-用户行为[实战]` 中的《用户行为-需求.md》《用户行为-接口文档.md》和补充资料《jackson进行序列化和反序列化解决.md》网页预览整理。课程要求所有行为数据存入 Redis；点赞、阅读、不喜欢由新建的 `heima-leadnews-behavior` 处理，关注由 `heima-leadnews-user` 处理，收藏和详情行为回显由 `heima-leadnews-article` 处理。

下文“资料要求”“接口约定”来自实际读取的文档；“工程补充”“设计示意”用于帮助实现和复习，不代表参考代码已经如此实现。未读取参考项目、`jackson.zip`、视频或图片资源包，未启动 Redis、数据库、Nacos、Kafka 或课程服务，业务联调尚未执行。原文图片没有嵌入本篇，流程以文字说明。

资料：[黑马头条项目分享链接](https://pan.baidu.com/s/1u7PNExeoEsYGh5pAnSyocg?pwd=9987)。


## 1. 用户行为与埋点：本课解决什么问题

资料中，用户行为包括关注、点赞、不喜欢、收藏、阅读等。它们既是 APP 的互动功能，也是后续文章推荐、热点统计和大数据分析的输入。

**埋点**是针对特定行为或事件进行捕获、处理、发送的数据采集过程。例如按钮点击次数、阅读文章时长、视频观看时长。用户点了一次按钮，不只改变页面样式，还需要把行为与当前用户、目标对象关联起来。

| 行为 | 需要记录的关系/数据 | 用户可见结果 |
|---|---|---|
| 关注 | 当前用户与作者的关系 | 关注按钮状态、我的关注、作者粉丝 |
| 点赞 | 当前用户与文章等对象的关系 | 点赞按钮高亮，可取消 |
| 阅读 | 当前用户对文章的阅读次数 | 为统计和推荐提供输入 |
| 不喜欢 | 当前用户对文章的负反馈 | 后续减少或过滤同类型推荐 |
| 收藏 | 当前用户收藏的文章 | 收藏按钮状态、个人收藏记录 |
| 详情回显 | 查询上述个人关系 | 再次打开文章时恢复按钮状态 |

区分两类数据：**当前状态**回答“现在是否点赞/收藏”；**行为事件**回答“何时点赞、何时取消、阅读过几次”。只保存最终状态会丢失动作历史。课程明确 Redis 存储，本篇的持久化事件、消息方案是扩展讨论，不是本日已经实现的功能。

```text
APP上的用户动作
       ↓
网关认证与路由
       ↓
user / behavior / article 服务
       ↓
Redis记录用户与目标的关系或阅读次数
       ├── 详情页查询 → 按钮状态回显
       └── 后续聚合   → 热点统计、推荐输入
```


## 2. 微服务职责与前面课程的衔接

资料明确规定服务归属，不能因为“都属于行为”就把六个接口全部放入 behavior 服务。

| 服务模块 | 本日职责 | 原文给出的接口 |
|---|---|---|
| `heima-leadnews-behavior`（新建） | 点赞、阅读、不喜欢 | `/api/v1/likes_behavior`、`/read_behavior`、`/un_likes_behavior` |
| `heima-leadnews-user` | 关注、取消关注 | `/api/v1/user/user_follow` |
| `heima-leadnews-article` | 收藏、取消收藏、详情行为回显 | `/api/v1/collection_behavior`、`/api/v1/article/load_article_behavior` |

表中省略前缀的路径只为便于阅读，完整路径见第04节。资料未给出本日网关外部前缀、端口、启动配置或 Redis key 常量，实际工程应与现有路由和代码核对。

与前几天的关系：

- 第02天文章详情提供内容和静态页面，本日叠加当前用户的动态互动状态。
- 第04天审核及发布形成 APP 文章，行为指向 APP 文章标识，不应误用自媒体稿件 ID。
- 第06天上下架改变文章可见性，行为接口与详情回显也要处理文章不存在或不可见的情况。
- 第08天实名认证通过后开通自媒体账号，为“APP 用户也是作者”建立身份关系。
- 第10、11天目录分别是定时和实时热点计算，本日负责准备行为数据，本篇不提前编造其评分公式或消息主题。

工程补充：新模块接入项目时，核对公共 DTO、认证上下文、Redis 客户端、异常处理和网关路由。只新增 Controller 路径而没有注册路由，前端仍然无法访问。


## 3. 登录身份、作者身份与文章身份

资料明确指出：登录用户才能关注、点赞、收藏；文章详情行为回显也要求登录。作者是 APP 用户经实名认证、开通作者权限后形成的身份，作者必然关联 APP 中的用户。

```text
当前APP用户 userId
       │ 实名认证、开通自媒体账号
       ▼
作者身份 authorId / 自媒体账号
       │ 发布后形成APP文章
       ▼
APP文章 articleId
```

**作者属于 APP 用户，不代表 `authorId` 一定数值等于 `userId`。** 作者、APP 用户和自媒体账号可能使用不同主键；关注关系要先按项目模型查明映射。粉丝关系中的“被关注人”也不能直接拿文章 ID 代替。

工程补充的请求处理顺序：

1. 从可信认证上下文读取当前用户，不接受客户端自填的操作人 ID。
2. 验证请求体、目标 ID、对象类型和操作值。
3. 验证文章、作者存在且可操作；同时校验文章与作者的关联。
4. 将作者身份映射到正确的 APP 用户或关系主体。
5. 按业务规则写 Redis，再返回统一响应。

以下概念示意并非资料中的完整源码：

```java
Long currentUserId = authenticatedUserId();
if (currentUserId == null) {
    return unauthorizedResult();
}
// 操作人来自认证上下文，articleId/authorId只是目标对象。
```

关注自己、已删除作者、下架文章是否允许取消历史行为，需要明确业务规则。资料未说明匿名阅读的身份标识策略，不能凭接口表出现 401 就断言匿名阅读一定支持或一定禁止。


## 4. 六个接口总表与统一响应

六个接口均使用 **POST**，请求体为 **`application/json`**，响应包装为 `ResponseResult`。

| 功能 | 完整路径 | DTO/请求字段 | 操作约定 |
|---|---|---|---|
| 点赞 | `/api/v1/likes_behavior` | `LikesBehaviorDto`：articleId、operation、type | operation：0点赞、1取消；type：0文章、1动态、2评论 |
| 阅读 | `/api/v1/read_behavior` | articleId、count | count为阅读次数 |
| 不喜欢 | `/api/v1/un_likes_behavior` | `UnLikesBehaviorDto`：articleId、type | type：0不喜欢、1取消不喜欢 |
| 关注 | `/api/v1/user/user_follow` | `UserRelationDto`：articleId、authorId、operation | operation：0关注、1取消 |
| 收藏 | `/api/v1/collection_behavior` | `CollectionBehaviorDto`：entryId、operation、publishedTime、type | operation：0收藏、1取消；type：0文章、1动态 |
| 详情回显 | `/api/v1/article/load_article_behavior` | `ArticleInfoDto`：articleId、authorId | 返回四个个人关系布尔值 |

容易混淆的地方：

- 不喜欢接口的 `type` 表示操作，点赞和收藏接口的 `type` 表示对象类型。
- 收藏目标字段叫 `entryId`，其他文章相关接口主要叫 `articleId`。
- `authorId` 在接口文档中为 `int`，`articleId`/`entryId` 为 `long`。
- 文档中 DTO 请求体标记为必需，许多 DTO 内部字段却标为 `false`。Swagger 表格不能替代业务必填校验。

响应模型包含 `code`、`data`、`errorMessage`、`host`。前五个接口的模板示例使用 `code: 0`，详情回显的具体示例使用 `code: 200`、`errorMessage: "操作成功"`。这是原文不同示例的差异，实际成功码应以项目 `ResponseResult` 和前端判断为准，不能把模板里的 0 自动定义为所有接口的真实成功码。

原文列出 HTTP 响应状态 200、201、401、403、404；HTTP 状态和 JSON 内业务码是两个层次。没有源码和联调依据时，应保留这种区别。


## 5. Redis 数据结构设计（工程补充）

资料只规定“所有行为数据存 Redis”，没有指定 key 名称、Hash/Set/ZSet、过期时间或 Redis 拓扑。下表是一套帮助理解的设计，不是对参考项目常量的复刻。

| 逻辑数据 | 结构示意 | 成员/字段 | 适用操作 |
|---|---|---|---|
| 对文章等对象的点赞 | Set：`behavior:likes:<type>:<targetId>` | userId | SADD、SREM、SISMEMBER、SCARD |
| 对文章的不喜欢 | Set：`behavior:unlikes:<articleId>` | userId | SADD、SREM、SISMEMBER |
| 文章的个人阅读次数 | Hash：`behavior:reads:<articleId>` | field=userId，value=次数 | HINCRBY、HGET |
| 我的关注 | Set：`user:following:<userId>` | 被关注APP用户ID | SADD、SREM、SISMEMBER |
| 我的粉丝 | Set：`user:followers:<targetUserId>` | 关注者userId | SADD、SREM、SCARD |
| 我的收藏 | ZSet：`article:collections:<userId>:<type>` | member=entryId，score=收藏时间 | ZADD NX、ZREM、ZSCORE、分页 |

选择理由：

- Set 天然去重，适合“是否已经操作”和关系数量；一次重复点赞不会多出第二个成员。
- Hash 可以按用户原子增加阅读次数，不必读取整个文章的行为对象。
- ZSet 可同时完成收藏去重与排序；收藏时间、文章发布时间是不同字段，不能混用。
- 若需要记录创建时间、来源、设备等详情，可补一个元数据 Hash 或使用明确版本的 JSON；不能只靠 Set 成员表达所有信息。

```text
点赞关系：目标 → 多个用户
关注关系：用户 → 多个被关注人，同时被关注人 → 多个粉丝
收藏关系：用户 → 按时间排序的多个内容对象
阅读计数：文章 → 每个用户的次数
```

工程边界：热门文章的集合/Hash 可能变成热点大 key；个人行为集合也会持续增长。按类型、目标或用户分区，设置容量与归档策略，避免 `KEYS *` 或一次性返回全量成员。某一种结构的设计改变后，所有写入与回显接口必须同步调整。


## 6. 关注与取消关注

### 6.1 资料要求与接口

当前登录用户可以关注和取消关注作者。一个用户可关注多个作者，作者可拥有多个粉丝，对应“我的关注”和“我的粉丝”。

`POST /api/v1/user/user_follow`，DTO 为 `UserRelationDto`：

| 字段 | 文档类型 | 含义 |
|---|---|---|
| articleId | long | 文章 ID |
| authorId | int | 作者 ID |
| operation | short | 0关注，1取消 |

```json
{
  "articleId": 1001,
  "authorId": 20,
  "operation": 0
}
```

示例数字只演示结构。原文没有说明 `articleId` 在关注逻辑中的具体用途；可用于校验文章与作者关系，但这种解释属于工程补充。

### 6.2 关系维护示意

```text
U关注作者A
  ↓ 查明A对应的APP用户V
following:U 增加V
followers:V 增加U

U取消关注A
following:U 删除V
followers:V 删除U
```

重复关注和重复取消应保持同一最终状态；只有真正新增或删除关系时才改变统计。业务上通常禁止关注自己，并校验作者是否有效，这些边界并非原文已给出的细则。

两个方向的索引若只写成功一个，“我的关注”和“我的粉丝”就会矛盾。在单 Redis 实例上可用事务/Lua 将配对写入一起执行；Redis Cluster 中，两人的 key 通常不在同一个 slot，不能直接套用跨 key Lua 原子操作。集群设计需要明确主关系、派生索引、可靠更新与修复任务，或采用确实能同 slot 的分区方案。


## 7. 点赞与取消点赞

### 7.1 资料要求与接口

登录用户点击“赞”后记录行为，可以取消点赞。

`POST /api/v1/likes_behavior`，DTO 为 `LikesBehaviorDto`：

| 字段 | 文档类型 | 取值/含义 |
|---|---|---|
| articleId | long | 文章 ID（字段原名） |
| operation | short | 0点赞，1取消点赞 |
| type | short | 0文章，1动态，2评论 |

```json
{
  "articleId": 1001,
  "operation": 0,
  "type": 0
}
```

文档允许三种对象类型，但没有说明动态、评论场景下 `articleId` 是否复用为目标 ID。实现这两种类型前要核对实际 DTO 和前端用法，本篇重点记录文章点赞。

### 7.2 Set 方案示意

```text
operation=0 → SADD behavior:likes:0:1001 U
operation=1 → SREM behavior:likes:0:1001 U
回显        → SISMEMBER behavior:likes:0:1001 U
当前数量    → SCARD behavior:likes:0:1001
```

`SADD` 返回1表示本次新增，返回0表示成员已存在；`SREM` 返回1表示本次删除，返回0表示原本不存在。它们可用于判断本次是否发生状态变化。

错误写法是“每来一个点赞请求就 `INCR`，每来一个取消就 `DECR`”：网络重试和连续点击会让数量膨胀或变成负数。若维护额外计数，只有真实关系变化才加减，并让关系与计数在合适的原子边界内更新。直接从集合 `SCARD` 取数可以避免这一份冗余计数不一致，但仍需考虑热点性能。


## 8. 阅读行为与 count 的语义

### 8.1 资料要求与接口

用户查看某篇文章时，记录当前用户查看的次数；阅读时长、阅读比例、加载时长为非必要信息。

`POST /api/v1/read_behavior`，请求字段：

| 字段 | 文档类型 | 含义 |
|---|---|---|
| articleId | long | 文章 ID |
| count | int | 阅读次数 |

```json
{
  "articleId": 1001,
  "count": 1
}
```

**文档没有定义 `count` 是“本次增量”还是“截至当前的累计值”。** 这会直接影响存储方式：

| 约定 | 存储方式示意 | 重试风险 |
|---|---|---|
| 本次增量 | HINCRBY 用户字段 count | 重复请求会重复累加，需要事件去重 |
| 累计值 | 校验后按约定覆盖/版本更新 | 旧请求晚到可能覆盖新值 |

只有明确采用增量语义后，才能使用下面的计数示意：

```text
HINCRBY behavior:reads:1001 U 1
HGET    behavior:reads:1001 U
```

### 8.2 工程补充

阅读量也要定义口径：打开一次详情、停留达到阈值、同一会话有效阅读，还是同一用户每日去重阅读。PV、UV、有效阅读次数是不同指标。

建议由服务端限制增量上限、频率和目标合法性，不信任客户端提交的任意大数。增加 `eventId`、会话或时间窗口可用于去重，但属于接口扩展，不是原文已有字段。时长用统一单位，比例用明确范围，服务端校验异常上报；不要用接口模板中的 `count: 0` 当作实际有效阅读事件。

未登录阅读若要支持，需要单独定义匿名会话/设备策略，以及登录后的关联规则，不能伪造一个 APP 用户 ID 来混入个人行为。


## 9. 不喜欢与取消不喜欢

### 9.1 资料要求与接口

需求指出：用户点击不喜欢后，不再向当前用户推荐这一类型的文章。接口额外提供取消不喜欢操作。

`POST /api/v1/un_likes_behavior`，DTO 为 `UnLikesBehaviorDto`：

| 字段 | 文档类型 | 含义 |
|---|---|---|
| articleId | long | 文章 ID |
| type | short | 0不喜欢，1取消不喜欢 |

```json
{
  "articleId": 1001,
  "type": 0
}
```

此处 `type` 是操作开关，不能套用点赞接口的“0文章、1动态、2评论”。

Set 方案示意：

```text
type=0 → SADD behavior:unlikes:1001 U
type=1 → SREM behavior:unlikes:1001 U
回显    → SISMEMBER behavior:unlikes:1001 U
```

### 9.2 从记录行为到推荐过滤

记录“不喜欢文章1001”只是第一步。要实现“不推荐同类型文章”，还需要查询文章的分类、标签或频道，把反馈传递给推荐/列表筛选规则。

```text
U对文章1001点不喜欢
       ↓
保存文章级负反馈
       ↓
解析文章分类/标签（产品定义“类型”）
       ↓
推荐候选中按规则过滤或降低相应类型权重
```

原文没有规定“类型”究竟是频道、内容标签还是算法类别，也没有给出取消后的恢复策略。不要把“不喜欢单篇”直接当作“永久屏蔽整个频道”，需要与产品规则核对。

点赞与不喜欢是否互斥也未在资料中说明。若工程选择互斥，应在同一业务操作中清除另一状态，并定义并发时的覆盖规则；若允许两者独立，则回显也要尊重独立状态，不能自行隐藏其中一个。


## 10. 文章收藏与取消收藏

### 10.1 资料要求与接口

需求记录当前登录人收藏的文章；接口支持取消收藏。

`POST /api/v1/collection_behavior`，DTO 为 `CollectionBehaviorDto`：

| 字段 | 文档类型 | 含义 |
|---|---|---|
| entryId | long | 文章 ID（原字段名） |
| operation | short | 0收藏，1取消收藏 |
| publishedTime | date | 发布时间 |
| type | short | 0文章，1动态 |

```json
{
  "entryId": 1001,
  "operation": 0,
  "publishedTime": "2026-10-10T10:00:00+08:00",
  "type": 0
}
```

时间字符串只是带时区的设计示例，原文仅给出 `date` 类型和空字符串模板，未指定格式、时区或绑定规则，必须与服务端 Jackson 配置一致。

### 10.2 收藏存储示意

```text
operation=0 → ZADD article:collections:U:0 NX <服务端收藏时间> 1001
operation=1 → ZREM article:collections:U:0 1001
回显        → ZSCORE article:collections:U:0 1001 是否存在
```

使用 `NX` 可以让重复收藏不改变首次收藏时间。如果产品希望重复操作刷新时间，就要改为明确的另一种规则；不能在重试时无意重排整个收藏列表。

`publishedTime` 是文章发布时间，**不是用户的收藏时间**。服务端应核实文章元信息，避免客户端伪造排序依据。需要保留发布时间、类型等信息时，可用元数据结构补充；ZSet 的一个 score 不能同时代表所有时间字段。

“我的收藏列表”如何分页、文章下架后如何展示、收藏上限多少，资料没有给出单独接口。本篇只列出自然的设计边界，不把它们写成课程已实现的额外接口。


## 11. 文章详情：四种行为状态回显

### 11.1 资料要求与接口

APP 用户必须登录，判断当前用户是否关注文章作者、收藏文章、点赞文章、不喜欢文章。例如已经点赞时，点赞按钮高亮。

`POST /api/v1/article/load_article_behavior`，DTO 为 `ArticleInfoDto`：

```json
{
  "articleId": 1001,
  "authorId": 20
}
```

原文的具体响应示例：

```json
{
  "host": null,
  "code": 200,
  "errorMessage": "操作成功",
  "data": {
    "islike": false,
    "isunlike": false,
    "iscollection": false,
    "isfollow": false
  }
}
```

四个字段名是全小写，保留原约定，不擅自改成 `isLike`、`isFollow`。Java 布尔属性与 Jackson/Lombok 的命名规则可能改变 JSON 字段，必要时显式指定 `@JsonProperty`，并核对前端解析。

### 11.2 读取流程示意

```text
当前登录用户U + 文章1001
       ↓ 查明文章真实作者A及其APP用户V
       ├── 点赞集合是否包含U   → islike
       ├── 不喜欢集合是否包含U → isunlike
       ├── U的收藏是否含1001   → iscollection
       └── U的关注是否包含V    → isfollow
       ↓
返回四个明确的boolean
```

请求中的 `authorId` 必须和文章的真实作者一致，或直接从文章推导；否则客户端替换作者 ID 会得到错误关注状态。回显接口只读取关系，不应因进入详情就创建点赞/关注，也不能顺便清空现有行为。

可以用 Redis pipeline 减少多次请求的网络往返，但 pipeline 不提供这四项结果的原子快照。用户正在并发操作时，短暂差异要通过前端请求顺序、版本或明确一致性策略处理。


## 12. 静态文章内容与个人动态行为

第02天的文章正文可以用 Freemarker 静态化并放在 MinIO，内容对多个用户相同。点赞、收藏和关注属于个人状态，即使正文相同，不同用户也可能得到完全不同的按钮状态。

```text
打开文章1001
    ├── 加载静态正文：共同内容
    └── 携带登录身份请求行为回显：U的个人状态
                         ↓
                   渲染四个按钮
```

工程补充：

- 不把用户 U 的点赞状态固化到所有人共用的静态 HTML 中。
- 不让公共缓存把 U 的行为响应返给 V；个人状态接口避免共享缓存，或使用正确的身份隔离策略。
- 区分回显接口与阅读上报接口，加载四个按钮不等于完成一次有效阅读。
- 行为操作成功后更新页面，重新打开文章时再以服务端状态为准。
- 快速连续点击时可串行提交，或使用请求序号避免较旧响应覆盖较新状态。

乐观更新示意：点击收藏后立即改变按钮，在失败时恢复原状态，并提示操作失败；服务端必须仍然幂等。仅靠前端禁用按钮无法防止网络重试、多个设备或并发请求。

文章已下架或作者身份变更时，重新验证可见性与真实关系，避免静态正文仍可访问却继续接受不合法的新增行为。


## 13. 幂等、并发与原子更新（工程补充）

**幂等**：同一请求重复执行，不重复增加关系或错误改变最终状态。例如对同一文章点赞三次，当前关系仍只有一条；取消三次也不应让计数变成负数。

| 操作 | 天然可幂等的状态写法 | 额外注意 |
|---|---|---|
| 点赞/不喜欢/关注新增 | SADD | 只在返回1时算新增 |
| 取消 | SREM/ZREM | 删除不存在项不重复扣数 |
| 收藏 | ZADD NX | 保护首次时间与排序 |
| 阅读增量 | HINCRBY本身原子，但不幂等 | 另需事件去重 |
| 详情回显 | 只读查询 | 不产生关系写入 |

一个 Redis 命令是原子的，不代表多命令序列原子。错误流程：

```text
先查没有点赞 → 写点赞 → 点赞总数+1
```

两个并发请求都可能看到“没有点赞”。可采用 SADD 的返回值决定真实新增；若关系、计数与消息都要一致，还要明确事务边界和失败处理。

适合单实例或同 slot key 的 Lua 学习示意：

```lua
-- KEYS[1]：点赞集合；KEYS[2]：冗余计数Hash；ARGV[1]：用户ID。
local added = redis.call('SADD', KEYS[1], ARGV[1])
if added == 1 then
    redis.call('HINCRBY', KEYS[2], 'likes', 1)
end
return added
```

此示意要求计数初始值正确，且所有新增、删除路径都执行配套原子逻辑。Redis Cluster 两个 key 必须同 slot，例如使用同一 `{article:1001}` hash tag；Lua 的原子性不等于任意运行时错误都会自动回滚，所以 key 类型与参数应在修改前验证。

幂等不能自动解决“先点赞、马上取消，但网络把请求倒序送达”。需要请求串行、操作版本或服务端排序规则。课程原接口没有版本字段，这属于要另行定义的扩展。


## 14. Redis 的生命周期、故障与服务边界

课程要求行为数据全部存 Redis，因此它们不是“丢掉也无所谓的缓存”：关注和收藏丢失会直接改变用户体验，行为统计丢失则影响推荐和热点输入。

工程补充的检查点：

- **持久化与恢复**：根据可容忍的数据丢失窗口配置 RDB/AOF、复制与备份，并做恢复验证。
- **内存与淘汰**：关系数据是否允许淘汰需明确；不能使用会随意淘汰业务关系的策略而没有恢复来源。
- **过期时间**：点赞、收藏、关注不能随手设置短 TTL；阅读明细/事件可按统计周期归档，但窗口和保留期限要有依据。
- **热点与大 key**：热门文章成员过多时分区、限流、监控；批量查询或分页使用适合的扫描/索引方式。
- **不可用处理**：Redis 查询异常表示状态未知，不等同于用户没有点赞；避免返回全 false 覆盖真实页面状态。
- **数据修复**：双向关注关系、额外计数和推荐读模型需要可检查、可重建的策略。

资料未规定 Redis 库编号、连接密码、部署模式或真实 key 常量。本篇不生成可直接连接生产环境的配置。

三个服务共享 Redis 并不意味着可以随意直接修改彼此数据。课程范围内可统一行为 key 与序列化契约；更大系统应明确每类关系的写入服务和查询方式，避免 key 命名、类型或版本改变时造成跨服务隐性故障。


## 15. Jackson 补充资料：为什么 ID 要转字符串

资料目录提供《jackson进行序列化和反序列化解决.md》。核心要求：后端响应含 `id` 或自定义标记字段时，将数值转换为 String；前端请求 DTO 中的对应字段再转换成 Integer/Long。

关联本日接口：`articleId`、`entryId` 为 Long。Java Long 可以表达很大的整数，但 JavaScript Number 的安全整数上限是 `2^53 - 1 = 9007199254740991`。如果大 ID 以 JSON 数字发送，前端解析为 Number 时可能已经损失精度。

```json
{
  "articleId": "9007199254740993"
}
```

这里字符串保持每一位数字。若先解析成 Number 再 `String(...)`，转换时已经丢失的精度不能恢复。前端从接口接收、保存、传参都应将 ID 保持为字符串，避免 `Number(id)`、`parseInt(id)` 或数值计算。

ID 序列化只改变传输表示，服务端模型仍可使用 Long/Integer，并严格校验解析范围。计数、操作枚举、比例等普通数值不应因此全改为字符串。

原资料把相关类称为“混淆”“解密”，但其 `ConfusionSerializer` 示例实际只是 `value.toString()` 后写 JSON 字符串。这不构成加密、不隐藏 ID，也不能替代目标对象权限校验。


## 16. Jackson 原资料中的类与注册流程

补充文档给出以下类的代码说明，可放在 `leadnews-common`，注解放在 model 公共包。这里只归纳已读取的 Markdown，不声称读取了 `jackson.zip`。

| 类/注解 | 原资料作用 |
|---|---|
| `IdEncrypt` | 自定义 Jackson 注解，标记非 `id` 名称但需要转换的字段 |
| `ConfusionSerializer` | 序列化时将非空值写为字符串 |
| `ConfusionDeserializer` | 依据目标类型转 Long/Integer，其他分支调用 `IdsUtils.decryptLong` |
| `ConfusionSerializerModifier` | 找到 `id`（忽略大小写）或带注解的属性，指定序列化器 |
| `ConfusionDeserializerModifier` | 给相同目标属性替换反序列化器 |
| `ConfusionModule` | 注册上述两个 Modifier；配置未知属性处理 |
| `InitJacksonConfig` | 创建并注册带该模块的 ObjectMapper |

原注解定义：

```java
@JacksonAnnotation
@Retention(RetentionPolicy.RUNTIME)
@Target({ElementType.FIELD, ElementType.METHOD, ElementType.PARAMETER})
public @interface IdEncrypt {
}
```

资料中的 DTO 用法：

```java
@Data
public class ArticleInfoDto {
    @IdEncrypt
    Long articleId;
}
```

自动按名称转换的字段是 `id`；`articleId`、`authorId`、`entryId` 不是这个名字，所以需显式注解或调整筛选逻辑，不能假设任何带 `Id` 后缀的属性都会被当前代码自动处理。

```text
Spring自动配置加载 InitJacksonConfig
       ↓
ObjectMapper 注册 ConfusionModule
       ├── 序列化 Modifier → 对id/注解字段写字符串
       └── 反序列化 Modifier → 对同类字段恢复Long/Integer
```

课程通过 `META-INF/spring.factories` 中的 `EnableAutoConfiguration` 条目增加 `com.heima.common.jackson.InitJacksonConfig`。这是课程配置方式；迁移到其他 Spring Boot 版本时，应确认对应的自动配置注册机制，不能未经核对直接照搬。


## 17. Jackson 示例的边界与修正建议

补充文档的序列化器结构如下：

```java
public class ConfusionSerializer extends JsonSerializer<Object> {
    @Override
    public void serialize(Object value, JsonGenerator generator,
                          SerializerProvider provider) throws IOException {
        if (value != null) {
            generator.writeString(value.toString());
            return;
        }
        provider.defaultSerializeValue(value, generator);
    }
}
```

上面保留核心行为，省略原文 try/catch 的打印逻辑。原文反序列化器检查 `type.getTypeName()` 是否含 `Long`/`Integer` 后使用 `valueOf`；其他情况尝试 `IdsUtils.decryptLong`，捕获异常后回退。以下为阅读资料后的工程补充：

| 原示例做法 | 需要处理的问题 |
|---|---|
| 通过类型名称包含关系识别类型 | 应明确支持包装类型、基本类型与合法字段，避免误匹配 |
| 解析失败后宽泛回退 | 空字符串、非数字、超范围 ID 应给出明确参数错误，不应悄悄返回不相关值 |
| 其他类型进入解密分支 | 当前序列化示例未加密；解密依赖与正反转换契约需单独确认 |
| `new ObjectMapper()` 替换默认实例 | 可能遗漏 Spring Boot 已注册的日期、JavaTime、命名与其他模块 |
| 未知字段配置为不失败 | `FAIL_ON_UNKNOWN_PROPERTIES=false` 实际是忽略未知字段，不是“抛错” |

若只是特定 Long 字段需要在响应中转字符串，可使用更窄的序列化配置，下面是**替代方案示意，不是课程原实现**：

```java
public class ArticleView {
    @JsonSerialize(using = ToStringSerializer.class)
    private Long articleId;
}
```

请求方向是否接受数字字符串应与 Jackson 的 coercion/严格模式配置核对，不能只测试响应就认为往返已经正确。也要校验 date 字段、空值、列表、Feign 和消息对象是否保持原来的配置。

优先扩展现有 Spring 管理的 ObjectMapper/模块配置，而不是在公共模块中无条件创建一个会覆盖全局的全新实例。全局 ID 转换会影响前七八天的文章、搜索和审核接口，修改时要核对这些契约。


## 18. 参数校验与错误处理（工程补充）

这一天的输入看似简单，字段混用却很容易让正确按钮操作写入错误关系。

| 输入/状态 | 应检查的内容 |
|---|---|
| 请求体 | 非空，字段名正确，支持的 JSON 类型 |
| 操作人 | 登录身份有效，从服务端认证上下文读取 |
| articleId/entryId | 非空、合法范围、指向正确APP内容对象 |
| authorId | 有效作者，并与文章关联；映射正确APP用户 |
| operation | 仅支持对应接口的0/1 |
| type | 按接口分别验证，不能共用错误枚举 |
| count | 先明确增量/累计语义，再限制值与频率 |
| publishedTime | 格式/时区一致，必要时从文章记录核实 |
| Redis key/value | 类型、版本和序列化格式一致 |

推荐错误分类：未登录返回认证错误；字段非法返回参数错误；目标不存在/不可见按项目规则返回目标错误；Redis 超时返回可重试的服务错误。具体状态码和 `ResponseResult` 枚举以工程现有约定为准。

不要把 Redis 查询失败吞掉后统一返回“未点赞”；否则用户可能重新点赞、取消或收藏，而页面状态也会被错误覆盖。回显的“确实没有关系”和“无法查询关系”必须能区分。

行为记录属于个人偏好数据，日志只写必要的业务标识与错误上下文，避免记录完整 token、设备指纹或全量用户画像。该建议不要求本次配置或运行任何外部系统。


## 19. 为热点与推荐准备数据（工程补充）

课程开篇强调埋点为热点推荐提供准备。本日只读到行为需求和接口，并没有热点评分公式、统计窗口、Kafka topic 或消费代码。

为了理解后续数据链路，可将行为变化抽象成事件：

```json
{
  "eventId": "<UNIQUE_EVENT_ID>",
  "userId": "<USER_ID>",
  "targetId": "<ARTICLE_ID>",
  "targetType": "article",
  "action": "like",
  "operation": "add",
  "occurredAt": "2026-10-10T10:00:00+08:00"
}
```

这个模型是额外设计，不是原接口新增字段。它帮助区分“当前点赞人数”与“在统计窗口内发生的点赞事件数”。取消操作是否扣回热度、重复阅读是否计入、历史反馈是否过期，应在后续统计规则中明确。

```text
关系状态 → 页面回显 / 当前关系数
变化事件 → 按窗口聚合 / 推荐特征 / 热点输入
```

若扩展为“先更新 Redis，再发送 Kafka”，需处理 Redis 成功但消息发送失败；若先发消息再写 Redis，则可能消费已发生但关系未更新。可靠事件记录、补偿和幂等消费是需要另外设计的环节，不能因为前面学过 Kafka 就把两个独立操作当作一个事务。

不喜欢反馈要能对应到文章类型；关注反馈要能对应到作者；收藏和阅读还可能使用不同权重。权重大小、公式和推荐算法留待实际后续资料，不在本篇猜测。


## 20. 实现顺序与联调清单（本次未执行）

### 20.1 建议实现顺序

1. 整理 DTO、操作枚举与统一响应，处理 articleId/entryId 的字符串传输契约。
2. 新建 behavior 模块并接入注册、路由、认证与 Redis 客户端。
3. 先实现点赞和不喜欢，验证去重、取消与状态查询。
4. 在 user 服务实现作者身份映射、双向关注关系。
5. 在 article 服务实现收藏及四项状态回显。
6. 明确 count 语义后实现阅读上报及去重规则。
7. 联调静态详情页与动态按钮，最后检查异常和并发场景。

### 20.2 核心验证场景

| 场景 | 应验证的结果 |
|---|---|
| 用户U点赞两次 | 只有一条当前关系；冗余计数不重复增加 |
| U取消两次 | 不存在点赞关系；数量不为负 |
| U关注作者A | U的关注与A的粉丝关系对应 |
| 作者ID不同于APP用户ID | 映射正确，没有把两个主键当同一个 |
| U收藏文章后重新打开 | iscollection为true，首次收藏时间按约定保存 |
| 不喜欢、取消不喜欢 | type=0/1正确，回显isunlike相应变化 |
| U和V查看同一静态文章 | 正文相同，四项行为状态按用户隔离 |
| 回显接口调用两次 | 不新增任何点赞、收藏或关注 |
| 大于2^53-1的ID | 前端以字符串保存，服务端还原后每位一致 |
| 阅读请求重试 | 按定义的语义和去重规则处理，不无意重复计数 |
| 作者参数被替换 | 检测与文章作者不匹配，不返回伪造关系 |
| Redis不可用 | 返回可识别错误，不伪造全false |
| 快速点赞后取消、响应乱序 | 页面不会被旧响应恢复成错误状态 |

这些是学习联调计划，不是本次执行结果。本次只校验笔记文件结构、编码、单元ID和代码围栏，未调用课程业务接口。


## 21. 常见问题与排查顺序

| 现象 | 优先排查 |
|---|---|
| 接口404 | 网关前缀、路由、Controller路径、behavior服务是否注册 |
| 按钮成功后再进详情不高亮 | 写入与回显使用的key、类型、用户ID是否一致 |
| 所有用户看到相同状态 | Redis key/成员没有身份隔离，或公共缓存了个人响应 |
| 关注列表和粉丝列表不一致 | 双向索引部分更新、作者与APP用户映射错误 |
| 点赞数越点越大 | 重复请求每次都增计数，没有依据关系变化判断 |
| 取消后统计出现负数 | 关系不存在时仍执行扣数 |
| 大文章ID查不到 | 前端Number精度损失、未注解articleId/entryId |
| 回显响应字段缺失 | Lombok/Jackson布尔属性命名与四个原字段不一致 |
| 收藏时间错乱 | 混用发布时间与收藏时间、时区不一致或重试刷新排序 |
| 阅读数翻倍 | count语义未统一或同一事件重复上报 |
| 点不喜欢仍看到同类型文章 | 只记录文章关系，未接入推荐过滤或“类型”定义不一致 |
| 引入Jackson配置后日期解析失败 | 新ObjectMapper覆盖了现有模块与格式配置 |

排查顺序：先确认真实请求路径和登录身份，再核对 DTO 字段/枚举/ID 精度，然后检查 Redis key、成员类型、写入返回值，接着验证作者映射与回显查询，最后看重试、并发、缓存和推荐侧更新。

Redis 中看到一条记录，不代表整个功能已经完成；要同时核对写入主体、目标对象、取消结果和重新进入详情页的状态。


## 22. 复习主线与自测

### 22.1 一条主线

```text
登录身份 → 校验目标 → 记录Redis行为 → 查询个人状态 → 详情按钮回显
                                   └→ 后续统计/推荐输入
```

- behavior：点赞、阅读、不喜欢；user：关注；article：收藏、详情行为回显。
- 原文全部行为存 Redis，但没有规定 key 和数据结构，示意方案与参考实现要区分。
- 点赞/关注/收藏用 operation=0新增、1取消；不喜欢使用 type=0/1，含义不同。
- 收藏请求叫 entryId；详情返回 islike、isunlike、iscollection、isfollow。
- 当前状态与事件历史不同；幂等关系写入不等于阅读事件幂等。
- 静态正文共享，动态关系按用户隔离；不能将个人状态写进公共静态页。
- Long ID 在 JSON 中按字符串传输解决前端精度问题，这不是加密。

### 22.2 自测题与参考答案

1. **为什么点赞、收藏都是关系，仍放在不同服务？** 课程规定按业务职责拆分：behavior处理通用行为，article管理文章收藏与详情聚合，user管理用户关系。
2. **不喜欢请求中的type=1是什么意思？** 取消不喜欢；不是“动态”类型。
3. **关注作者时为什么不能直接使用authorId当APP用户ID？** 作者身份与APP用户有对应关系，但主键不一定相同，需要明确映射。
4. **SADD为什么有助于避免重复点赞？** 同一成员只保存一次，返回值还能指示本次是否新增；额外计数必须据此更新。
5. **HINCRBY是原子的，为什么阅读重试仍会重复计数？** 每次调用都原子增加一次，但同一事件两次调用仍增加两次，需要事件去重或其他明确定义。
6. **count到底是增量还是累计？** 当前接口文档未定义，必须在实现前核对；两者对应的更新策略不同。
7. **详情回显为什么要动态请求？** 文章内容可共享，个人关系因用户而异，不能写在共用静态HTML里。
8. **pipeline是否保证四项回显同时刻一致？** 不保证；它减少往返，不提供原子快照。
9. **为何大ID先转Number再转String不行？** 第一次Number解析可能已丢失精度，后续String不能恢复原数字。
10. **@IdEncrypt是否真正加密ID？** 原示例序列化器只是转字符串，注解名称不能当作加密或访问控制。
11. **Redis Cluster里能直接Lua更新任意两个人的关注key吗？** 通常不能；多key脚本要求同slot，需设计分区或可靠的派生索引更新方案。
12. **保存“不喜欢文章”是否已经实现“不推荐同类型”？** 没有；还要定义类型并让推荐/筛选链路使用反馈。

### 22.3 资料覆盖记录

| 已读取资料 | 本篇覆盖 |
|---|---|
| 用户行为-需求.md | 行为与埋点、关注、点赞、阅读、不喜欢、收藏、详情回显、Redis及三个服务分工 |
| 用户行为-接口文档.md | 六个POST路径、DTO字段、操作值、统一响应、四个原始回显字段 |
| jackson进行序列化和反序列化解决.md | ID转字符串、自定义注解、序列化/反序列化器、Modifier、Module、ObjectMapper与自动配置 |

未读取视频、参考代码、jackson.zip或图片资源包；笔记中的Redis结构、原子更新、可靠事件和联调计划均已标为工程补充。完成笔记不等于完成项目实现或业务测试。
